# 🌐 CPI Bangladesh Mission — Google Drive OS Builder
### `000_BGD_CPMS` — Complete Folder, Sub-Folder & File Structure

---

**Version:** v1.0 | **Date:** 2026-04-20 | **Author:** ariful@cpintl.org  
**Shared Drive:** `000_BGD_CPMS`  
**Drive URL:** https://drive.google.com/drive/folders/18N_9Eq4oOCPZoTBW4CbsVNqLVBkvb6KW

---

## What this notebook does

This notebook **automatically builds** the complete CPI Bangladesh Mission Digital Operating System inside your `000_BGD_CPMS` Google Shared Drive — including:

- ✅ All folders and sub-folders (4-tier hierarchy)
- ✅ All Google Sheets (tracker templates)
- ✅ All Google Docs (SOP templates, report shells)
- ✅ Template Registry master index (version control ledger)
- ✅ JSON schema stubs for every form/sheet
- ✅ Proper CPI naming conventions throughout
- ✅ Version tagging on all master files

## Hierarchy built
```
Global HQ → Country Mission (000_BGD_CPMS) → Project Office (CoxsBazar) → Field Sites
```

## ⚠️ Before running
1. Make sure you are signed in as **ariful@cpintl.org**
2. Run cells **in order** — top to bottom
3. When prompted to authorise Google Drive access, click **Allow**
4. The notebook is **safe to re-run** — it checks if folders already exist before creating them

---

## STEP 1 — Install dependencies & authenticate

In [ ]:
# Install required libraries
!pip install --quiet google-api-python-client google-auth-httplib2 google-auth-oauthlib

print("✅ Libraries installed successfully.")

In [ ]:
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import json, time

# Build the Drive service
drive_service = build('drive', 'v3')

print("✅ Authenticated and Drive service ready.")

## STEP 2 — Configuration: your Shared Drive ID

In [ ]:
# ============================================================
#  CONFIGURATION — edit only this cell if needed
# ============================================================

# Your 000_BGD_CPMS Shared Drive ID
# Extracted from: https://drive.google.com/drive/folders/18N_9Eq4oOCPZoTBW4CbsVNqLVBkvb6KW
SHARED_DRIVE_ID = "18N_9Eq4oOCPZoTBW4CbsVNqLVBkvb6KW"

# Root label (for logging only — this IS your Shared Drive root)
ROOT_LABEL = "000_BGD_CPMS"

# Date stamp for file naming
TODAY = "2026-04"

# Version for all master templates
VER = "v1.0"

print(f"✅ Configuration set.")
print(f"   Shared Drive ID : {SHARED_DRIVE_ID}")
print(f"   Root label      : {ROOT_LABEL}")
print(f"   Date stamp      : {TODAY}")
print(f"   Version         : {VER}")

## STEP 3 — Core helper functions

In [ ]:
# ============================================================
#  HELPER FUNCTIONS
# ============================================================

created_log = []   # tracks every item created this run
skipped_log = []   # tracks items that already existed


def find_item(name, parent_id, mime_type=None):
    """Return the ID of an existing file/folder, or None if not found."""
    q = f"name='{name}' and '{parent_id}' in parents and trashed=false"
    if mime_type:
        q += f" and mimeType='{mime_type}'"
    try:
        results = drive_service.files().list(
            q=q,
            spaces='drive',
            fields='files(id, name)',
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()
        files = results.get('files', [])
        return files[0]['id'] if files else None
    except HttpError:
        return None


def create_folder(name, parent_id):
    """Create a folder; skip if already exists. Returns folder ID."""
    existing = find_item(name, parent_id, 'application/vnd.google-apps.folder')
    if existing:
        skipped_log.append(f"📁 [EXISTS] {name}")
        return existing
    meta = {
        'name': name,
        'mimeType': 'application/vnd.google-apps.folder',
        'parents': [parent_id]
    }
    folder = drive_service.files().create(
        body=meta,
        fields='id',
        supportsAllDrives=True
    ).execute()
    fid = folder['id']
    created_log.append(f"📁 [CREATED] {name}")
    time.sleep(0.15)   # gentle rate limit
    return fid


def create_gsheet(name, parent_id):
    """Create a blank Google Sheet; skip if already exists. Returns file ID."""
    existing = find_item(name, parent_id, 'application/vnd.google-apps.spreadsheet')
    if existing:
        skipped_log.append(f"📊 [EXISTS] {name}")
        return existing
    meta = {
        'name': name,
        'mimeType': 'application/vnd.google-apps.spreadsheet',
        'parents': [parent_id]
    }
    f = drive_service.files().create(
        body=meta,
        fields='id',
        supportsAllDrives=True
    ).execute()
    created_log.append(f"📊 [CREATED] {name}")
    time.sleep(0.15)
    return f['id']


def create_gdoc(name, parent_id):
    """Create a blank Google Doc; skip if already exists. Returns file ID."""
    existing = find_item(name, parent_id, 'application/vnd.google-apps.document')
    if existing:
        skipped_log.append(f"📄 [EXISTS] {name}")
        return existing
    meta = {
        'name': name,
        'mimeType': 'application/vnd.google-apps.document',
        'parents': [parent_id]
    }
    f = drive_service.files().create(
        body=meta,
        fields='id',
        supportsAllDrives=True
    ).execute()
    created_log.append(f"📄 [CREATED] {name}")
    time.sleep(0.15)
    return f['id']


def create_json_stub(name, parent_id, schema_dict):
    """Upload a JSON schema file; skip if already exists."""
    existing = find_item(name + '.json', parent_id)
    if existing:
        skipped_log.append(f"🗂️  [EXISTS] {name}.json")
        return existing
    import io
    from googleapiclient.http import MediaIoBaseUpload
    content = json.dumps(schema_dict, indent=2).encode('utf-8')
    meta = {'name': name + '.json', 'parents': [parent_id]}
    media = MediaIoBaseUpload(io.BytesIO(content), mimetype='application/json')
    f = drive_service.files().create(
        body=meta, media_body=media,
        fields='id', supportsAllDrives=True
    ).execute()
    created_log.append(f"🗂️  [CREATED] {name}.json")
    time.sleep(0.15)
    return f['id']


def section(title):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")


print("✅ Helper functions loaded.")

## STEP 4 — Build the complete folder structure

This is the main builder. It creates every folder in the correct hierarchy.

In [ ]:
# ============================================================
#  FOLDER STRUCTURE BUILDER
#  Root = SHARED_DRIVE_ID (this IS 000_BGD_CPMS)
# ============================================================

ROOT = SHARED_DRIVE_ID
ids = {}   # stores all folder IDs by key

section("LAYER 1 — Root-level folders (Mission layer)")

ids['00_TPL']      = create_folder("00_Templates",  ROOT)
ids['01_HRA']      = create_folder("01_HRA",        ROOT)
ids['02_GF']       = create_folder("02_G&F",        ROOT)
ids['03_LSC']      = create_folder("03_LSC",        ROOT)
ids['04_MER']      = create_folder("04_MER",        ROOT)
ids['05_MC']       = create_folder("05_MC",         ROOT)
ids['10_PROG']     = create_folder("10_Programs",   ROOT)
ids['11_PROJ']     = create_folder("11_Projects",   ROOT)
ids['99_ARC']      = create_folder("99_Archive",    ROOT)

print("\n✅ Root-level folders done.")

# -----------------------------------------------------------
section("LAYER 2 — 00_Templates subfolders")

ids['TPL_Forms']   = create_folder("Forms",         ids['00_TPL'])
ids['TPL_Sheets']  = create_folder("Sheets",        ids['00_TPL'])
ids['TPL_Docs']    = create_folder("Docs",          ids['00_TPL'])
ids['TPL_JSON']    = create_folder("JSON_Schemas",  ids['00_TPL'])
ids['TPL_VER']     = create_folder("VERSION_HISTORY", ids['00_TPL'])

print("\n✅ 00_Templates subfolders done.")

# -----------------------------------------------------------
section("LAYER 2 — 01_HRA subfolders")

ids['HRA_HRM']     = create_folder("HRM",           ids['01_HRA'])
ids['HRA_ADMIN']   = create_folder("Admin",         ids['01_HRA'])
ids['HRA_RECR']    = create_folder("Recruitment",   ids['01_HRA'])
ids['HRA_LEAVE']   = create_folder("Leave_Records", ids['01_HRA'])

print("✅ 01_HRA done.")

# -----------------------------------------------------------
section("LAYER 2 — 02_G&F subfolders")

ids['GF_BUDG']     = create_folder("Budgets",         ids['02_GF'])
ids['GF_GRANT']    = create_folder("Grants",          ids['02_GF'])
ids['GF_REPORT']   = create_folder("Finance_Reports", ids['02_GF'])
ids['GF_REIMB']    = create_folder("Reimbursements",  ids['02_GF'])

print("✅ 02_G&F done.")

# -----------------------------------------------------------
section("LAYER 2 — 03_LSC subfolders")

ids['LSC_GL']      = create_folder("General_Logistics", ids['03_LSC'])
ids['LSC_MEDLOG']  = create_folder("Medical_Logistics", ids['03_LSC'])
ids['LSC_PROC']    = create_folder("Procurement",       ids['03_LSC'])
ids['LSC_WH']      = create_folder("Warehouse",         ids['03_LSC'])

print("✅ 03_LSC done.")

# -----------------------------------------------------------
section("LAYER 2 — 04_MER subfolders")

ids['MER_FW']      = create_folder("Frameworks",      ids['04_MER'])
ids['MER_IND']     = create_folder("Indicators",      ids['04_MER'])
ids['MER_REP']     = create_folder("Reports",         ids['04_MER'])
ids['MER_DASH']    = create_folder("Dashboards",      ids['04_MER'])
ids['MER_DQA']     = create_folder("Data_Quality",    ids['04_MER'])

print("✅ 04_MER done.")

# -----------------------------------------------------------
section("LAYER 2 — 05_MC subfolders")

ids['MC_BRAND']    = create_folder("Brand",           ids['05_MC'])
ids['MC_PHOTO']    = create_folder("Photos_Videos",   ids['05_MC'])
ids['MC_PUB']      = create_folder("Publications",    ids['05_MC'])

print("✅ 05_MC done.")

# -----------------------------------------------------------
section("LAYER 2 — 10_Programs subfolders (mission frameworks only)")

ids['PROG_HN']     = create_folder("H&N",  ids['10_PROG'])
ids['PROG_SD']     = create_folder("S&D",  ids['10_PROG'])
ids['PROG_ERP']    = create_folder("ERP",  ids['10_PROG'])
ids['PROG_RL']     = create_folder("R&L",  ids['10_PROG'])

print("✅ 10_Programs done.")

# -----------------------------------------------------------
section("LAYER 2 — 11_Projects: Cox's Bazar Project Office")

ids['CXB']         = create_folder("CoxsBazar",       ids['11_PROJ'])

# Cox's Bazar mirrors mission support units
ids['CXB_TPL']     = create_folder("00_Templates",    ids['CXB'])
ids['CXB_HRA']     = create_folder("01_HRA",          ids['CXB'])
ids['CXB_GF']      = create_folder("02_G&F",          ids['CXB'])
ids['CXB_LSC']     = create_folder("03_LSC",          ids['CXB'])
ids['CXB_MER']     = create_folder("04_MER",          ids['CXB'])
ids['CXB_MC']      = create_folder("05_MC",           ids['CXB'])

# CXB Templates subfolders
ids['CXB_TPL_F']   = create_folder("Forms",           ids['CXB_TPL'])
ids['CXB_TPL_S']   = create_folder("Sheets",          ids['CXB_TPL'])
ids['CXB_TPL_D']   = create_folder("Docs",            ids['CXB_TPL'])
ids['CXB_TPL_J']   = create_folder("JSON_Schemas",    ids['CXB_TPL'])

# CXB support unit subfolders
ids['CXB_HRA_HRM'] = create_folder("HRM",             ids['CXB_HRA'])
ids['CXB_HRA_ADM'] = create_folder("Admin",           ids['CXB_HRA'])
ids['CXB_LSC_GL']  = create_folder("General_Logistics", ids['CXB_LSC'])
ids['CXB_LSC_ML']  = create_folder("Medical_Logistics", ids['CXB_LSC'])
ids['CXB_LSC_PR']  = create_folder("Procurement",     ids['CXB_LSC'])
ids['CXB_LSC_WH']  = create_folder("Warehouse",       ids['CXB_LSC'])
ids['CXB_MER_IND'] = create_folder("Indicators",      ids['CXB_MER'])
ids['CXB_MER_REP'] = create_folder("Reports",         ids['CXB_MER'])

print("\n✅ CoxsBazar support units done.")

# -----------------------------------------------------------
section("LAYER 3 — CoxsBazar Program folders")

# H&N
ids['CXB_HN']      = create_folder("H&N",             ids['CXB'])
ids['CXB_HOP']     = create_folder("HOP",             ids['CXB_HN'])
ids['CXB_HPP']     = create_folder("HPP",             ids['CXB_HN'])
ids['CXB_HSS']     = create_folder("HSS",             ids['CXB_HN'])

# S&D
ids['CXB_SD']      = create_folder("S&D",             ids['CXB'])
ids['CXB_WASH']    = create_folder("WASH",            ids['CXB_SD'])
ids['CXB_CE']      = create_folder("CE",              ids['CXB_SD'])
ids['CXB_EDU']     = create_folder("EDU",             ids['CXB_SD'])
ids['CXB_LIV']     = create_folder("LIV",             ids['CXB_SD'])

# ERP and R&L
ids['CXB_ERP']     = create_folder("ERP",             ids['CXB'])
ids['CXB_RL']      = create_folder("R&L",             ids['CXB'])

# Partners
ids['CXB_PART']    = create_folder("Partners",        ids['CXB'])
ids['YPSA']        = create_folder("YPSA",            ids['CXB_PART'])
ids['EDUCO']       = create_folder("Educo",           ids['CXB_PART'])
ids['SDA']         = create_folder("SDA",             ids['CXB_PART'])
ids['RTMI']        = create_folder("RTMI",            ids['CXB_PART'])
ids['ICDDRB']      = create_folder("ICDDRB_Friendship", ids['CXB_PART'])

print("\n✅ CoxsBazar program folders done.")

# -----------------------------------------------------------
section("LAYER 4 — Field Sites (Camp-level folders)")

# HOP field sites
ids['HOP_01W']     = create_folder("Camp_01W",        ids['CXB_HOP'])
ids['HOP_01W_A']   = create_folder("Block_A",         ids['HOP_01W'])
ids['HOP_01W_B']   = create_folder("Block_B",         ids['HOP_01W'])
ids['HOP_04']      = create_folder("Camp_04",         ids['CXB_HOP'])
ids['HOP_04_D']    = create_folder("Block_D",         ids['HOP_04'])
ids['HOP_04_F']    = create_folder("Block_F",         ids['HOP_04'])

# HPP field site
ids['HPP_01W_B']   = create_folder("Camp_01W_Block_B", ids['CXB_HPP'])

# HSS field sites
ids['HSS_SADAR']   = create_folder("Sadar_Hospital",      ids['CXB_HSS'])
ids['HSS_RTMI']    = create_folder("RTMI_PHCC_Camp05",    ids['CXB_HSS'])

# WASH field sites
ids['WASH_01W']    = create_folder("Camp_01W",        ids['CXB_WASH'])
ids['WASH_01W_B']  = create_folder("Block_B",         ids['WASH_01W'])
ids['WASH_01W_G']  = create_folder("Block_G",         ids['WASH_01W'])
ids['WASH_04']     = create_folder("Camp_04",         ids['CXB_WASH'])
ids['WASH_04_D']   = create_folder("Block_D",         ids['WASH_04'])
ids['WASH_17']     = create_folder("Camp_17",         ids['CXB_WASH'])
ids['WASH_HOST']   = create_folder("Host_Community",  ids['CXB_WASH'])
ids['WASH_DAR']    = create_folder("Dariyanagar_Asroyon", ids['WASH_HOST'])

# Archive sub-structure
create_folder("HRA_Archive",      ids['99_ARC'])
create_folder("GF_Archive",       ids['99_ARC'])
create_folder("LSC_Archive",      ids['99_ARC'])
create_folder("Programs_Archive", ids['99_ARC'])
create_folder("Projects_Archive", ids['99_ARC'])

print("\n✅ All field site folders done.")
print(f"\n{'='*60}")
print(f"  FOLDER STRUCTURE COMPLETE")
print(f"  Created : {len([l for l in created_log if '📁' in l])} folders")
print(f"  Skipped : {len([l for l in skipped_log if '📁' in l])} already existed")
print(f"{'='*60}")

## STEP 5 — Create master Google Sheets (tracker templates)

In [ ]:
# ============================================================
#  MASTER GOOGLE SHEETS — stored in 00_Templates/Sheets
# ============================================================

section("Creating master Google Sheets in 00_Templates/Sheets")

sheets_to_create = [
    # (display name, parent folder key)
    (f"CPI_BGD_TemplateRegistry_Master_{VER}",            '00_TPL'),    # registry in root templates
    (f"CPI_BGD_HRA_Sheet_StaffDirectory_{VER}",           'TPL_Sheets'),
    (f"CPI_BGD_HRA_Sheet_LeaveTracker_{TODAY}_{VER}",     'TPL_Sheets'),
    (f"CPI_BGD_GF_Sheet_BudgetTracker_{TODAY}_{VER}",     'TPL_Sheets'),
    (f"CPI_BGD_GF_Sheet_ExpenseTracker_{TODAY}_{VER}",    'TPL_Sheets'),
    (f"CPI_BGD_GF_Sheet_PettyCashLog_{TODAY}_{VER}",      'TPL_Sheets'),
    (f"CPI_BGD_LSC_Sheet_InventoryTracker_{VER}",         'TPL_Sheets'),
    (f"CPI_BGD_LSC_Sheet_ProcurementTracker_{VER}",       'TPL_Sheets'),
    (f"CPI_BGD_LSC_Sheet_VehicleLog_{VER}",               'TPL_Sheets'),
    (f"CPI_BGD_MedLog_Sheet_StockRegister_{VER}",         'TPL_Sheets'),
    (f"CPI_BGD_MER_Sheet_IndicatorTracker_{TODAY}_{VER}", 'TPL_Sheets'),
    (f"CPI_BGD_MER_Sheet_DataQualityChecklist_{VER}",     'TPL_Sheets'),
    (f"CPI_BGD_HN_Sheet_PatientSummary_{VER}",            'TPL_Sheets'),
    (f"CPI_BGD_HOP_Sheet_DailyPatientLog_{VER}",          'TPL_Sheets'),
    (f"CPI_BGD_HPP_Sheet_OPDRegister_{VER}",              'TPL_Sheets'),
    (f"CPI_BGD_HSS_Sheet_ActivityTracker_{VER}",          'TPL_Sheets'),
    (f"CPI_BGD_WASH_Sheet_InfraTracker_{VER}",            'TPL_Sheets'),
    (f"CPI_BGD_WASH_Sheet_MonthlyMonitoring_{VER}",       'TPL_Sheets'),
    (f"CPI_BGD_ERP_Sheet_IncidentLog_{VER}",              'TPL_Sheets'),
    (f"CPI_BGD_RL_Sheet_ResearchDataLog_{VER}",           'TPL_Sheets'),
]

sheet_ids = {}
for name, parent_key in sheets_to_create:
    pid = ids.get(parent_key, ids['00_TPL'])
    sheet_ids[name] = create_gsheet(name, pid)
    print(f"   {name}")

print(f"\n✅ {len(sheets_to_create)} master Sheets created/verified.")

## STEP 6 — Create master Google Docs (SOP & report templates)

In [ ]:
# ============================================================
#  MASTER GOOGLE DOCS — stored in 00_Templates/Docs
# ============================================================

section("Creating master Google Docs in 00_Templates/Docs")

docs_to_create = [
    # SOPs
    f"CPI_BGD_SOP_Template_Shell_{VER}",
    f"CPI_BGD_HRA_SOP_Recruitment_{TODAY}_{VER}",
    f"CPI_BGD_GF_SOP_Finance_{TODAY}_{VER}",
    f"CPI_BGD_LSC_SOP_Procurement_{TODAY}_{VER}",
    f"CPI_BGD_MER_SOP_DataCollection_{TODAY}_{VER}",
    f"CPI_BGD_HRA_SOP_Safeguarding_{TODAY}_{VER}",
    # Report templates
    f"CPI_BGD_MER_Doc_MonthlyReport_Template_{VER}",
    f"CPI_BGD_MER_Doc_QuarterlyReport_Template_{VER}",
    f"CPI_BGD_MER_Doc_AnnualReport_Template_{VER}",
    # Meeting & admin templates
    f"CPI_BGD_Admin_Doc_MeetingMinutes_Template_{VER}",
    f"CPI_BGD_HRA_Doc_HandoverNote_Template_{VER}",
    f"CPI_BGD_HRA_Doc_ExitInterview_Template_{VER}",
    f"CPI_BGD_HRA_Doc_ProbationEvaluation_Template_{VER}",
    f"CPI_BGD_HRA_Doc_EmployeeHandbook_2026-01_{VER}",
    # Program docs
    f"CPI_BGD_HN_Doc_Framework_{VER}",
    f"CPI_BGD_SD_Doc_Framework_{VER}",
    f"CPI_BGD_ERP_Doc_ContingencyPlan_{VER}",
    f"CPI_BGD_RL_Doc_ResearchProtocol_{VER}",
    f"CPI_BGD_MER_Doc_MELFramework_{VER}",
    f"CPI_BGD_MER_Doc_ReportingCalendar_2026_{VER}",
    f"CPI_BGD_MC_Doc_BrandGuidelines_{VER}",
]

for name in docs_to_create:
    create_gdoc(name, ids['TPL_Docs'])
    print(f"   {name}")

print(f"\n✅ {len(docs_to_create)} master Docs created/verified.")

## STEP 7 — Create JSON schema stubs

In [ ]:
# ============================================================
#  JSON SCHEMA STUBS — stored in 00_Templates/JSON_Schemas
# ============================================================

section("Creating JSON schema files in 00_Templates/JSON_Schemas")

def schema(template_name, program, unit, fields):
    return {
        "schema_version": "1.0",
        "template_name": template_name,
        "program": program,
        "unit": unit,
        "created": TODAY,
        "fields": fields
    }

json_schemas = [
    (
        f"CPI_BGD_HOP_JSON_DailyPatientLogSchema_{VER}",
        schema("HOP Daily Patient Log", "H&N", "HOP", [
            {"name": "Record_ID",       "type": "string",   "required": True},
            {"name": "Date",            "type": "date",     "required": True},
            {"name": "Camp",            "type": "dropdown", "options": ["Camp_01W", "Camp_04"], "required": True},
            {"name": "Block",           "type": "dropdown", "options": ["A", "B", "D", "F"],   "required": True},
            {"name": "Total_Patients",  "type": "integer",  "required": True},
            {"name": "New_Patients",    "type": "integer",  "required": True},
            {"name": "Female",          "type": "integer",  "required": True},
            {"name": "Under5",          "type": "integer",  "required": True},
            {"name": "Pregnant_Women",  "type": "integer",  "required": False},
            {"name": "Referrals",       "type": "integer",  "required": False},
            {"name": "Staff_Name",      "type": "string",   "required": True},
            {"name": "Notes",           "type": "text",     "required": False}
        ])
    ),
    (
        f"CPI_BGD_HPP_JSON_OPDRegisterSchema_{VER}",
        schema("HPP OPD Register", "H&N", "HPP", [
            {"name": "Patient_ID",        "type": "string",   "required": True},
            {"name": "Visit_Date",        "type": "date",     "required": True},
            {"name": "Age",               "type": "integer",  "required": True},
            {"name": "Sex",               "type": "dropdown", "options": ["M", "F"], "required": True},
            {"name": "Origin_Camp",       "type": "string",   "required": True},
            {"name": "Chief_Complaint",   "type": "text",     "required": True},
            {"name": "Diagnosis_ICD10",   "type": "string",   "required": False},
            {"name": "Treatment_Given",   "type": "text",     "required": True},
            {"name": "Referred_Out",      "type": "boolean",  "required": False},
            {"name": "Staff_Name",        "type": "string",   "required": True}
        ])
    ),
    (
        f"CPI_BGD_WASH_JSON_MonthlyMonitoringSchema_{VER}",
        schema("WASH Monthly Monitoring", "S&D", "WASH", [
            {"name": "Record_ID",              "type": "string",   "required": True},
            {"name": "Reporting_Month",        "type": "date",     "required": True},
            {"name": "Camp",                   "type": "dropdown", "options": ["Camp_01W", "Camp_04", "Camp_17", "Host_Community"], "required": True},
            {"name": "Block",                  "type": "string",   "required": True},
            {"name": "Latrines_Built",         "type": "integer",  "required": True},
            {"name": "Latrines_Functional",    "type": "integer",  "required": True},
            {"name": "Safe_Water_Points",      "type": "integer",  "required": True},
            {"name": "HH_Trainings",           "type": "integer",  "required": False},
            {"name": "Issues_Noted",           "type": "text",     "required": False},
            {"name": "Staff_Name",             "type": "string",   "required": True}
        ])
    ),
    (
        f"CPI_BGD_GF_JSON_ExpenseClaimSchema_{VER}",
        schema("Expense Claim", "All", "G&F", [
            {"name": "Claim_ID",        "type": "string",   "required": True},
            {"name": "Staff_Name",      "type": "string",   "required": True},
            {"name": "Department",      "type": "string",   "required": True},
            {"name": "Expense_Date",    "type": "date",     "required": True},
            {"name": "Category",        "type": "dropdown", "options": ["Travel", "Accommodation", "Per_Diem", "Office_Supplies", "Other"], "required": True},
            {"name": "Amount_BDT",      "type": "number",   "required": True},
            {"name": "Description",     "type": "text",     "required": True},
            {"name": "Receipt_Attached","type": "boolean",  "required": True}
        ])
    ),
    (
        f"CPI_BGD_LSC_JSON_ProcurementRequestSchema_{VER}",
        schema("Procurement Request", "All", "LSC", [
            {"name": "Request_ID",      "type": "string",   "required": True},
            {"name": "Requested_By",    "type": "string",   "required": True},
            {"name": "Unit",            "type": "string",   "required": True},
            {"name": "Item_Name",       "type": "string",   "required": True},
            {"name": "Quantity",        "type": "integer",  "required": True},
            {"name": "Unit_of_Measure", "type": "string",   "required": True},
            {"name": "Needed_By_Date",  "type": "date",     "required": True},
            {"name": "Justification",   "type": "text",     "required": True},
            {"name": "Budget_Line",     "type": "string",   "required": True}
        ])
    ),
    (
        f"CPI_BGD_MER_JSON_IndicatorTrackerSchema_{VER}",
        schema("MER Indicator Tracker", "All", "MER", [
            {"name": "Indicator_ID",        "type": "string",   "required": True},
            {"name": "Indicator_Name",      "type": "string",   "required": True},
            {"name": "Program",             "type": "dropdown", "options": ["H&N", "S&D", "ERP", "R&L"], "required": True},
            {"name": "Unit",                "type": "string",   "required": True},
            {"name": "Annual_Target",       "type": "integer",  "required": True},
            {"name": "YTD_Actual",          "type": "integer",  "required": True},
            {"name": "Pct_Achieved",        "type": "formula",  "required": False},
            {"name": "Reporting_Month",     "type": "date",     "required": True},
            {"name": "Responsible_Staff",   "type": "string",   "required": True},
            {"name": "Last_Updated",        "type": "date",     "required": True}
        ])
    ),
    (
        f"CPI_BGD_HRA_JSON_StaffDirectorySchema_{VER}",
        schema("Staff Directory", "All", "HRA", [
            {"name": "Staff_ID",       "type": "string",   "required": True},
            {"name": "Full_Name",      "type": "string",   "required": True},
            {"name": "Department",     "type": "string",   "required": True},
            {"name": "Program",        "type": "dropdown", "options": ["H&N", "S&D", "ERP", "R&L", "HRA", "G&F", "LSC", "MER", "MC"], "required": True},
            {"name": "Location",       "type": "dropdown", "options": ["Dhaka", "CoxsBazar", "Field"], "required": True},
            {"name": "Supervisor",     "type": "string",   "required": True},
            {"name": "Email",          "type": "email",    "required": True},
            {"name": "Contract_Type",  "type": "dropdown", "options": ["Full-time", "Part-time", "Consultant", "Intern"], "required": True},
            {"name": "Start_Date",     "type": "date",     "required": True},
            {"name": "Status",         "type": "dropdown", "options": ["Active", "On_Leave", "Separated"], "required": True}
        ])
    ),
]

for fname, fdata in json_schemas:
    create_json_stub(fname, ids['TPL_JSON'], fdata)
    print(f"   {fname}.json")

print(f"\n✅ {len(json_schemas)} JSON schema files created/verified.")

## STEP 8 — Populate the Template Registry master sheet

Writes column headers and one row per master template into the central index sheet using the Google Sheets API.

In [ ]:
# ============================================================
#  POPULATE TEMPLATE REGISTRY SHEET
# ============================================================

from googleapiclient.discovery import build as gbuild

sheets_api = gbuild('sheets', 'v4')

# Find the registry sheet ID
registry_name = f"CPI_BGD_TemplateRegistry_Master_{VER}"
registry_id = find_item(registry_name, ids['00_TPL'], 'application/vnd.google-apps.spreadsheet')

if not registry_id:
    print("❌ Registry sheet not found. Run Step 5 first.")
else:
    # Build registry rows
    header = [
        "Template_ID", "Template_Name", "Unit", "Doc_Type",
        "Version", "Date_Created", "Drive_Folder", "File_Type",
        "JSON_Schema_Filename", "Status", "Updated_By", "Notes"
    ]

    registry_rows = [
        ["T001", f"CPI_BGD_HRA_Sheet_StaffDirectory_{VER}",           "HRA",  "Sheet", "1.0", TODAY, "00_Templates/Sheets", "Google Sheet", f"CPI_BGD_HRA_JSON_StaffDirectorySchema_{VER}.json",     "Active", "ariful@cpintl.org", "Master staff database"],
        ["T002", f"CPI_BGD_HRA_Sheet_LeaveTracker_{TODAY}_{VER}",     "HRA",  "Sheet", "1.0", TODAY, "00_Templates/Sheets", "Google Sheet", "",                                                       "Active", "ariful@cpintl.org", "Annual leave tracker"],
        ["T003", f"CPI_BGD_GF_Sheet_BudgetTracker_{TODAY}_{VER}",     "G&F",  "Sheet", "1.0", TODAY, "00_Templates/Sheets", "Google Sheet", "",                                                       "Active", "ariful@cpintl.org", "Project budget tracker"],
        ["T004", f"CPI_BGD_GF_Sheet_ExpenseTracker_{TODAY}_{VER}",    "G&F",  "Sheet", "1.0", TODAY, "00_Templates/Sheets", "Google Sheet", f"CPI_BGD_GF_JSON_ExpenseClaimSchema_{VER}.json",         "Active", "ariful@cpintl.org", "Expense log"],
        ["T005", f"CPI_BGD_GF_Sheet_PettyCashLog_{TODAY}_{VER}",      "G&F",  "Sheet", "1.0", TODAY, "00_Templates/Sheets", "Google Sheet", "",                                                       "Active", "ariful@cpintl.org", "Petty cash daily log"],
        ["T006", f"CPI_BGD_LSC_Sheet_InventoryTracker_{VER}",         "LSC",  "Sheet", "1.0", TODAY, "00_Templates/Sheets", "Google Sheet", "",                                                       "Active", "ariful@cpintl.org", "Warehouse inventory"],
        ["T007", f"CPI_BGD_LSC_Sheet_ProcurementTracker_{VER}",       "LSC",  "Sheet", "1.0", TODAY, "00_Templates/Sheets", "Google Sheet", f"CPI_BGD_LSC_JSON_ProcurementRequestSchema_{VER}.json",  "Active", "ariful@cpintl.org", "Procurement status"],
        ["T008", f"CPI_BGD_LSC_Sheet_VehicleLog_{VER}",               "LSC",  "Sheet", "1.0", TODAY, "00_Templates/Sheets", "Google Sheet", "",                                                       "Active", "ariful@cpintl.org", "Vehicle movement log"],
        ["T009", f"CPI_BGD_MedLog_Sheet_StockRegister_{VER}",         "LSC",  "Sheet", "1.0", TODAY, "00_Templates/Sheets", "Google Sheet", "",                                                       "Active", "ariful@cpintl.org", "Medical drug stock"],
        ["T010", f"CPI_BGD_MER_Sheet_IndicatorTracker_{TODAY}_{VER}", "MER",  "Sheet", "1.0", TODAY, "00_Templates/Sheets", "Google Sheet", f"CPI_BGD_MER_JSON_IndicatorTrackerSchema_{VER}.json",    "Active", "ariful@cpintl.org", "Program KPI tracker"],
        ["T011", f"CPI_BGD_HOP_Sheet_DailyPatientLog_{VER}",          "H&N",  "Sheet", "1.0", TODAY, "00_Templates/Sheets", "Google Sheet", f"CPI_BGD_HOP_JSON_DailyPatientLogSchema_{VER}.json",     "Active", "ariful@cpintl.org", "HOP patient count"],
        ["T012", f"CPI_BGD_HPP_Sheet_OPDRegister_{VER}",              "H&N",  "Sheet", "1.0", TODAY, "00_Templates/Sheets", "Google Sheet", f"CPI_BGD_HPP_JSON_OPDRegisterSchema_{VER}.json",         "Active", "ariful@cpintl.org", "Health post OPD"],
        ["T013", f"CPI_BGD_WASH_Sheet_MonthlyMonitoring_{VER}",       "S&D",  "Sheet", "1.0", TODAY, "00_Templates/Sheets", "Google Sheet", f"CPI_BGD_WASH_JSON_MonthlyMonitoringSchema_{VER}.json",   "Active", "ariful@cpintl.org", "WASH camp monitoring"],
        ["T014", f"CPI_BGD_ERP_Sheet_IncidentLog_{VER}",              "ERP",  "Sheet", "1.0", TODAY, "00_Templates/Sheets", "Google Sheet", "",                                                       "Active", "ariful@cpintl.org", "Emergency incidents"],
        ["T015", f"CPI_BGD_SOP_Template_Shell_{VER}",                 "All",  "Doc",   "1.0", TODAY, "00_Templates/Docs",   "Google Doc",  "",                                                       "Active", "ariful@cpintl.org", "Blank SOP shell"],
        ["T016", f"CPI_BGD_MER_Doc_MonthlyReport_Template_{VER}",     "MER",  "Doc",   "1.0", TODAY, "00_Templates/Docs",   "Google Doc",  "",                                                       "Active", "ariful@cpintl.org", "Monthly report shell"],
        ["T017", f"CPI_BGD_MER_Doc_QuarterlyReport_Template_{VER}",   "MER",  "Doc",   "1.0", TODAY, "00_Templates/Docs",   "Google Doc",  "",                                                       "Active", "ariful@cpintl.org", "Quarterly report shell"],
        ["T018", f"CPI_BGD_Admin_Doc_MeetingMinutes_Template_{VER}",  "Admin","Doc",   "1.0", TODAY, "00_Templates/Docs",   "Google Doc",  "",                                                       "Active", "ariful@cpintl.org", "Meeting minutes shell"],
        ["T019", f"CPI_BGD_HN_Doc_Framework_{VER}",                   "H&N",  "Doc",   "1.0", TODAY, "00_Templates/Docs",   "Google Doc",  "",                                                       "Active", "ariful@cpintl.org", "H&N mission framework"],
        ["T020", f"CPI_BGD_MER_Doc_MELFramework_{VER}",               "MER",  "Doc",   "1.0", TODAY, "00_Templates/Docs",   "Google Doc",  "",                                                       "Active", "ariful@cpintl.org", "MEL framework"],
    ]

    values = [header] + registry_rows

    sheets_api.spreadsheets().values().update(
        spreadsheetId=registry_id,
        range="Sheet1!A1",
        valueInputOption="RAW",
        body={"values": values}
    ).execute()

    print(f"✅ Template Registry populated with {len(registry_rows)} rows.")
    print(f"   Open: https://docs.google.com/spreadsheets/d/{registry_id}")

## STEP 9 — Print the complete summary report

In [ ]:
# ============================================================
#  FINAL SUMMARY
# ============================================================

total_created = len(created_log)
total_skipped = len(skipped_log)

folders_c = len([x for x in created_log if '📁' in x])
sheets_c  = len([x for x in created_log if '📊' in x])
docs_c    = len([x for x in created_log if '📄' in x])
json_c    = len([x for x in created_log if '🗂️' in x])

print("")
print("╔══════════════════════════════════════════════════════════╗")
print("║   CPI Bangladesh Mission Digital OS — BUILD COMPLETE     ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║   Shared Drive : 000_BGD_CPMS                            ║")
print(f"║   Run date     : {TODAY}                                ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║   📁 Folders created   : {folders_c:<32}║")
print(f"║   📊 Sheets created    : {sheets_c:<32}║")
print(f"║   📄 Docs created      : {docs_c:<32}║")
print(f"║   🗂️  JSON schemas      : {json_c:<32}║")
print(f"║   ⏭️  Items skipped     : {total_skipped:<32}║")
print(f"║   ✅ Total items       : {total_created + total_skipped:<32}║")
print("╚══════════════════════════════════════════════════════════╝")
print("")
print("🔗 Open your Shared Drive:")
print(f"   https://drive.google.com/drive/folders/{SHARED_DRIVE_ID}")
print("")
print("🔗 Open the Template Registry:")
if registry_id:
    print(f"   https://docs.google.com/spreadsheets/d/{registry_id}")
print("")
print("── HIERARCHY BUILT ──────────────────────────────────────")
print("Global HQ → 000_BGD_CPMS (Mission) → CoxsBazar (Project) → Field Sites")
print("")
print("── NEXT STEPS ───────────────────────────────────────────")
print("1. Open the Template Registry and review all entries")
print("2. Set folder permissions (see Permissions tab in Claude)")
print("3. Add column headers to each master Google Sheet")
print("4. Create Google Forms and link them to response Sheets")
print("5. Set up Looker Studio dashboard using Indicator Tracker")
print("─────────────────────────────────────────────────────────")

## STEP 10 — (Optional) Add column headers to all master Sheets

Run this cell to automatically write the correct column headers into every master Sheet tracker.

In [ ]:
# ============================================================
#  WRITE COLUMN HEADERS TO ALL MASTER SHEETS
# ============================================================

# Map: (sheet name, parent folder key) → column headers
sheet_headers = {
    f"CPI_BGD_HRA_Sheet_StaffDirectory_{VER}": [
        "Staff_ID","Full_Name","Department","Program","Location",
        "Supervisor","Email","Contract_Type","Start_Date","Status"
    ],
    f"CPI_BGD_HRA_Sheet_LeaveTracker_{TODAY}_{VER}": [
        "Staff_ID","Full_Name","Leave_Type","Start_Date","End_Date",
        "Days_Requested","Days_Approved","Approved_By","Status","Notes"
    ],
    f"CPI_BGD_GF_Sheet_BudgetTracker_{TODAY}_{VER}": [
        "Budget_Line_ID","Project","Donor","Category","Approved_Budget_BDT",
        "Spent_to_Date_BDT","Remaining_BDT","Pct_Spent","Reporting_Period","Notes"
    ],
    f"CPI_BGD_GF_Sheet_ExpenseTracker_{TODAY}_{VER}": [
        "Claim_ID","Staff_Name","Department","Expense_Date","Category",
        "Amount_BDT","Description","Receipt_Attached","Approved_By","Status"
    ],
    f"CPI_BGD_GF_Sheet_PettyCashLog_{TODAY}_{VER}": [
        "Date","Description","Receipt_No","Debit_BDT","Credit_BDT",
        "Balance_BDT","Category","Authorized_By"
    ],
    f"CPI_BGD_LSC_Sheet_InventoryTracker_{VER}": [
        "Item_ID","Item_Name","Category","Unit","Opening_Stock",
        "Received","Issued","Closing_Balance","Expiry_Date","Warehouse_Location"
    ],
    f"CPI_BGD_LSC_Sheet_ProcurementTracker_{VER}": [
        "Request_ID","Requested_By","Unit","Item_Name","Quantity",
        "Unit_of_Measure","Needed_By_Date","Budget_Line","Status","Notes"
    ],
    f"CPI_BGD_LSC_Sheet_VehicleLog_{VER}": [
        "Trip_ID","Date","Vehicle_Reg","Driver","Destination",
        "Purpose","Departure_KM","Return_KM","Distance_KM","Authorized_By"
    ],
    f"CPI_BGD_MedLog_Sheet_StockRegister_{VER}": [
        "Item_ID","Drug_Name","Form","Strength","Opening_Stock",
        "Received_Qty","Dispensed_Qty","Closing_Balance","Expiry_Date","Batch_No"
    ],
    f"CPI_BGD_MER_Sheet_IndicatorTracker_{TODAY}_{VER}": [
        "Indicator_ID","Indicator_Name","Program","Unit","Annual_Target",
        "YTD_Actual","Pct_Achieved","Reporting_Month","Responsible_Staff","Last_Updated"
    ],
    f"CPI_BGD_MER_Sheet_DataQualityChecklist_{VER}": [
        "Check_ID","Data_Source","Program","Checked_By","Check_Date",
        "Completeness","Accuracy","Timeliness","Issues_Found","Action_Taken"
    ],
    f"CPI_BGD_HN_Sheet_PatientSummary_{VER}": [
        "Reporting_Month","Program","Camp","Total_Patients","New_Patients",
        "Female","Under5","Pregnant_Women","Referrals_Out","Notes"
    ],
    f"CPI_BGD_HOP_Sheet_DailyPatientLog_{VER}": [
        "Record_ID","Date","Camp","Block","Total_Patients",
        "New_Patients","Female","Under5","Pregnant_Women","Referrals","Staff_Name","Notes"
    ],
    f"CPI_BGD_HPP_Sheet_OPDRegister_{VER}": [
        "Patient_ID","Visit_Date","Age","Sex","Origin_Camp",
        "Chief_Complaint","Diagnosis_ICD10","Treatment_Given","Referred_Out","Staff_Name"
    ],
    f"CPI_BGD_HSS_Sheet_ActivityTracker_{VER}": [
        "Activity_ID","Date","Site","Activity_Type","Target_Beneficiaries",
        "Actual_Beneficiaries","Staff_Involved","Notes","Reporting_Month"
    ],
    f"CPI_BGD_WASH_Sheet_InfraTracker_{VER}": [
        "Asset_ID","Asset_Type","Camp","Block","Installation_Date",
        "Status","Last_Inspection","Condition","Repair_Needed","Notes"
    ],
    f"CPI_BGD_WASH_Sheet_MonthlyMonitoring_{VER}": [
        "Record_ID","Reporting_Month","Camp","Block","Latrines_Built",
        "Latrines_Functional","Safe_Water_Points","HH_Trainings","Issues_Noted","Staff_Name"
    ],
    f"CPI_BGD_ERP_Sheet_IncidentLog_{VER}": [
        "Incident_ID","Date","Location","Incident_Type","Description",
        "Affected_People","Response_Action","Reported_By","Status","Escalated_To"
    ],
    f"CPI_BGD_RL_Sheet_ResearchDataLog_{VER}": [
        "Study_ID","Study_Name","Lead_Partner","Start_Date","End_Date",
        "Status","Target_Sample","Enrolled","Ethics_Clearance","Notes"
    ],
}

print("Writing column headers to all master Sheets...\n")
ok_count = 0

for sheet_name, headers in sheet_headers.items():
    sid = find_item(sheet_name, ids['TPL_Sheets'], 'application/vnd.google-apps.spreadsheet')
    if not sid:
        # Also check root templates folder (for registry)
        sid = find_item(sheet_name, ids['00_TPL'], 'application/vnd.google-apps.spreadsheet')
    if sid:
        try:
            sheets_api.spreadsheets().values().update(
                spreadsheetId=sid,
                range="Sheet1!A1",
                valueInputOption="RAW",
                body={"values": [headers]}
            ).execute()
            print(f"   ✅ Headers written → {sheet_name}")
            ok_count += 1
            time.sleep(0.2)
        except Exception as e:
            print(f"   ⚠️  Could not write → {sheet_name}: {e}")
    else:
        print(f"   ⚠️  Not found → {sheet_name}")

print(f"\n✅ Headers written to {ok_count}/{len(sheet_headers)} sheets.")

---

## Reference: CPI Naming Convention

| Pattern | Example |
|---|---|
| `CPI_BGD_[UNIT]_[DOCTYPE]_[Name]_[YYYY-MM]_v[X.Y]` | `CPI_BGD_HOP_Sheet_DailyPatientLog_2026-04_v1.0` |
| `CPI_BGD_[PROGRAM]_[DOCTYPE]_[Name]_v[X.Y]` | `CPI_BGD_MER_Doc_MELFramework_v1.0` |
| Field working copy: `CPI_CXB_[PROG]_[Camp]_[Name]_[YYYY-MM]` | `CPI_CXB_HOP_Camp01W_PatientLog_2026-04` |

## Reference: Folder prefix order (always)

```
00_Templates   ← always first
01_HRA
02_G&F
03_LSC
04_MER
05_MC
10_Programs
11_Projects
99_Archive     ← always last
```

## Reference: Version rules

| Change | Rule | Example |
|---|---|---|
| First release | v1.0 | `v1.0` |
| Minor fix / field added | Increment decimal | `v1.0 → v1.1` |
| Major restructure | Increment whole | `v1.2 → v2.0` |
| Retired | Move to 99_Archive | `..._DEPRECATED_v1.2` |

---
*CPI Bangladesh Mission Digital OS — built with Google Drive API v3 + Sheets API v4*  
*Notebook version: v1.0 | ariful@cpintl.org*